# Imports and Setup

In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np

import jax
import jax.numpy as jnp
from jax import vmap, jit, grad, jacfwd

# Optional: higher numeric accuracy (do this before heavy JAX use)
# try:
#     from jax import config as jax_config
#     jax_config.update("jax_enable_x64", True)
# except Exception:
#     pass

import skewerjax as sjx
from skewerjax.constants.physical_constants import phys

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = sjx.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

In [ ]:
from skewerjax.skewers.skewers import SkewerBundle
from skewerjax.cosmology.cosmology import Cosmology

### Define some global variables that will be employed throughout

In [ ]:
# Create cosmology object
cosmo_Planck15 = Cosmology(
    H0=phys.H0,
    Omega_r_0=9.1436e-5,  # Radiation density (CMB + neutrinos)
    Omega_m_0=0.3089,
    Omega_k_0=0.0,
    Omega_lambda_0=0.6911,
)

In [ ]:
L_box = 1.0  # Gpc

_spatial_points_x = L_box * jnp.array([ # define spatial points along x corresponding to a single skewer
    [0.0, 0.5, 0.5],
    [0.2, 0.5, 0.5],
    [0.3, 0.5, 0.5],
    [1.0, 0.5, 0.5],
])

# Validation of skewers checking different stored attributes

In [ ]:
# Define spatial points for multiple skewers
spatial_points = jnp.array([
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x
])

# Initial conditions per skewer
a0_per_skewer = 1.0                                               # scale factor at which ICs are defined, same for all skewers
x0_per_skewer = 0.0                                               # comoving distance at which ICs are defined, same for all skewers
ICs_indices = jnp.array([0, 0, -1, -1])                           # indices along the skewer where ICs are defined (0 if the ICs are defined at the first point, -1 if at the last point of the spatial_points array)
ICs_types = ["emission", "absorption", "emission", "absorption"]  # whether the ICs are defined at emission or absorption

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,                                # Let SkewerBundle compute los_directions automatically
    manual_scale_factors_in_the_skewer_direction=None,  # let SkewerBundle compute scale factors automatically
    cosmologies=cosmo_Planck15                          # when the manual scale factors are not provided, cosmologies must be defined
)

In [ ]:
print("Number of skewers (N_sk):", skewer_bundle.N_sk)
print("Number of points per skewer (N_ppsk):", skewer_bundle.N_ppsk)
print("Number of cosmologies (N_cosmologies):", cosmo_Planck15.N_cosmologies)

print("ICs defined at indices:", skewer_bundle.ICs_defined_at_index)
print("ICs are defined at absorption (if False, it is defined at emission):", skewer_bundle.is_absorption)
print("Array directions of skewers:", skewer_bundle.direction)

In [ ]:
oriented_spatial_points = skewer_bundle.oriented_spatial_points
print("Oriented spatial points shape:", oriented_spatial_points.shape, " - should be (N_sk, N_ppsk, 3)")
print(oriented_spatial_points)

In [ ]:
comoving_distances = skewer_bundle.comoving_distances_in_the_skewer_direction
print("Comoving distances in the skewer direction:", comoving_distances.shape, " - should be (N_sk, N_ppsk)")
print(comoving_distances)

In [ ]:
los_directions = skewer_bundle.los_directions
print("Line-of-sight directions:", los_directions.shape, " - should be (N_sk, N_ppsk, 3)")
print(los_directions)

In [ ]:
scale_factors = skewer_bundle.scale_factors_in_the_skewer_direction
print("scale_factors:", scale_factors.shape, " - should be (N_sk, N_cosmologies, N_ppsk)")
print(scale_factors)

### Check if the scale factors are correctly computed independent of invariant coordinate transformations

In [ ]:
spatial_points = jnp.array([
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x,
    _spatial_points_x
])

# Initial conditions per skewer
a0_per_skewer = jnp.array([1.0, 1.0, 0.8010958, 0.8010958])
x0_per_skewer = jnp.array([0.0, 0.0, L_box, 0.0])
ICs_indices = jnp.array([0, -1, -1, -1])
ICs_types = ["absorption", "absorption", "emission", "emission"]

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,
    cosmologies=cosmo_Planck15
)

In [ ]:
print("Comoving distances in the skewer direction:\n", skewer_bundle.comoving_distances_in_the_skewer_direction)
print("scale_factors:\n", skewer_bundle.scale_factors_in_the_skewer_direction)

### Check if los_directions are correctly computed 

In [ ]:
_spatial_points_zigzag = L_box * jnp.array([
    [0.0, 0.0, 0.0],
    [0.1, 0.2, 0.2],
    [0.3, 0.3, 0.3],
    [1.0, 1.0, 1.0],
])

In [ ]:
spatial_points = jnp.array([
    _spatial_points_zigzag,
    _spatial_points_zigzag,
    _spatial_points_zigzag,
    _spatial_points_zigzag
])

# Initial conditions per skewer
a0_per_skewer = 1.0
x0_per_skewer = 0.0
ICs_indices = jnp.array([0, 0, -1, -1])
ICs_types = ["emission", "absorption", "emission", "absorption"]

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,
    cosmologies=cosmo_Planck15
)

In [ ]:
print("Line-of-sight directions:\n", skewer_bundle.los_directions)

### Validation of scale factors computation employing several cosmologies

In [ ]:
cosmologies = Cosmology(
    H0             = jnp.array([phys.H0, phys.H0, phys.H0, phys.H0, phys.H0]),
    Omega_r_0      = jnp.array([1.0, 0.0, 0.0, 0.0, 9.1436e-5]),
    Omega_m_0      = jnp.array([0.0, 1.0, 0.0, 0.0, 0.3089]),
    Omega_k_0      = jnp.array([0.0, 0.0, 1.0, 0.0, 0.0]),
    Omega_lambda_0 = jnp.array([0.0, 0.0, 0.0, 1.0, 0.6911]),
)

# Define spatial points for multiple skewers
spatial_points = jnp.array([
    _spatial_points_x,
    _spatial_points_x
])

# Initial conditions per skewer
a0_per_skewer = jnp.array([0.8010958, 1.0])
x0_per_skewer = 0.0
ICs_indices = jnp.array([0, 0])
ICs_types = ["emission", "absorption"]

# Create SkewerBundle
skewer_bundle = SkewerBundle(
    spatial_points=spatial_points,
    a0=a0_per_skewer,
    x0=x0_per_skewer,
    ICs_defined_at_index=ICs_indices,
    ICs_at_emission_or_absorption=ICs_types,
    los_directions=None,
    cosmologies=cosmologies
)

scale_factors = skewer_bundle.scale_factors_in_the_skewer_direction
# show if dimensions coincide with expectations
print("Number of skewers (N_sk):", skewer_bundle.N_sk)
print("Number of points per skewer (N_ppsk):", skewer_bundle.N_ppsk)
print("Number of cosmologies (N_cosmologies):", cosmologies.N_cosmologies)
print("scale_factors dimensions:\n", scale_factors.shape, " - should be (N_sk, N_cosmologies, N_ppsk)")
print("scale_factors:\n", scale_factors)